In [1]:
# ----------------------------------------------------------------------
# 1. Imports and setup
# ----------------------------------------------------------------------
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel, RunnableBranch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
import os
import re
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
api_key = os.getenv("HUGGINGFACE_API_TOKEN")

# ----------------------------------------------------------------------
# 2. Initialize model and vector store
# ----------------------------------------------------------------------
# Model
llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=api_key,
)
model = ChatHuggingFace(llm=llm)

# Embeddings
os.environ["HF_HOME"] = "/Users/kristalshrestha/Documents/Code/LLM_Scratch/models"  # optional
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Vector store (must already be populated)
vector_store = Chroma(
    embedding_function=hf_embeddings,
    persist_directory="./2VectorDatabaseChroma",
    collection_name="gedsi_collection",
)

# Fixed refusal message
FIXED_REFUSAL = (
    "I'm sorry, I can only answer questions related to Gender Equality, "
    "Disability, and Social Inclusion (GEDSI) in Nepal. Please ask a question "
    "about legal rights, discrimination, or government procedures in these areas."
)

# ----------------------------------------------------------------------
# 3. Pydantic models for structured outputs
# ----------------------------------------------------------------------
class GEDSILegalAnswer(BaseModel):
    verdict: str = Field(description="Short conclusion on whether the action is legal/illegal")
    law_says: str = Field(description="Relevant legal provisions (act, section, text)")
    punishment: str = Field(description="Penalties if applicable")
    what_to_do: str = Field(description="Step‑by‑step recommended actions")

class GEDSIClassification(BaseModel):
    is_gedsi: bool = Field(description="Whether the query is about GEDSI topics in Nepal")

# ----------------------------------------------------------------------
# 4. Helper functions (wrapped as RunnableLambda later)
# ----------------------------------------------------------------------
def retrieve_best(query: str, k: int = 1):
    """Return the best document and its distance."""
    docs_with_scores = vector_store.similarity_search_with_score(query, k=k)
    if not docs_with_scores:
        return None, None
    doc, score = docs_with_scores[0]
    return doc, score

def is_gedsi_classifier(query: str) -> bool:
    """Classify query using structured output."""
    parser = PydanticOutputParser(pydantic_object=GEDSIClassification)
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a strict classifier. GEDSI stands for Gender Equality, Disability, and Social Inclusion.\n"
         "A question is GEDSI-related if it involves discrimination, rights, or issues based on gender, disability, ethnicity, caste, sexual orientation, or social inclusion in Nepal.\n"
         "Output only a valid JSON object with a single field 'is_gedsi'. {format_instructions}"),
        ("human", "Question: A woman is denied citizenship because she is female.\nAnswer:"),
        ("assistant", '{{"is_gedsi": true}}'),
        ("human", "Question: What is the capital of France?\nAnswer:"),
        ("assistant", '{{"is_gedsi": false}}'),
        ("human", "Question: I am a woman and the government officer refused to issue me a citizenship certificate.\nAnswer:"),
        ("assistant", '{{"is_gedsi": true}}'),
        ("human", "Question: {query}\nAnswer:")
    ])
    chain = prompt | model
    raw_response = chain.invoke({
        "query": query,
        "format_instructions": parser.get_format_instructions()
    })
    content = raw_response.content.strip()
    # Try to parse whole content
    try:
        return parser.parse(content).is_gedsi
    except Exception:
        # Try regex extraction
        json_pattern = r'\{.*?\}'
        matches = re.findall(json_pattern, content, re.DOTALL)
        for match in matches:
            try:
                return parser.parse(match).is_gedsi
            except Exception:
                continue
        # Fallback
        return False

def generate_legal_answer_structured(query: str) -> str:
    """Generate structured legal answer."""
    parser = PydanticOutputParser(pydantic_object=GEDSILegalAnswer)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a legal assistant specialized in Nepali law. Answer the user's question in a structured way using the provided format.\n{format_instructions}"),
        ("human", "{query}")
    ])
    chain = prompt | model | parser
    try:
        result = chain.invoke({
            "query": query,
            "format_instructions": parser.get_format_instructions()
        })
        return f"**Verdict:** {result.verdict}\n\n**Law says:** {result.law_says}\n\n**Punishment:** {result.punishment}\n\n**What to do:** {result.what_to_do}"
    except Exception as e:
        return f"Error generating answer: {e}. Please refine your question."

def branch_a_logic(inputs):
    """Branch A: high similarity (distance <= threshold) – use stored answer."""
    doc = inputs["doc"]
    doc_type = doc.metadata.get("type")
    if doc_type == "refusal":
        return FIXED_REFUSAL
    elif doc_type == "actual":
        return doc.metadata.get("answer", "No answer found.")
    else:
        # Unknown type – fallback to Branch B
        return branch_b_logic(inputs)

def branch_b_logic(inputs):
    """Branch B: low similarity – classify and generate or refuse."""
    query = inputs["query"]
    if is_gedsi_classifier(query):
        return generate_legal_answer_structured(query)
    else:
        return FIXED_REFUSAL

# ----------------------------------------------------------------------
# 5. Build the chain
# ----------------------------------------------------------------------
# Step 1: Retrieve and attach query, doc, score
retrieve_runnable = RunnableLambda(lambda q: (q, *retrieve_best(q))) | RunnableLambda(
    lambda t: {"query": t[0], "doc": t[1], "score": t[2] if t[1] else None}
)

# Step 2: Branch based on score
DISTANCE_THRESHOLD = 0.55
branch_condition = lambda x: x["score"] is not None and x["score"] <= DISTANCE_THRESHOLD

# Define the two branches as runnables
branch_a = RunnableLambda(branch_a_logic)
branch_b = RunnableLambda(branch_b_logic)

# Main conditional chain
conditional_chain = retrieve_runnable | RunnableBranch(
    (branch_condition, branch_a),
    branch_b  # default
)

# ----------------------------------------------------------------------
# 6. Test the chain (optional)
# ----------------------------------------------------------------------
if __name__ == "__main__":
    test_queries = [
        "I am a woman and the government officer refused to issue me a citizenship certificate.",
        "I am a woman and the government officer refused to issue me a citizenship certificate because I am a woman, saying only men can get it. Is this allowed under Nepal law?",
        "What is the capital of France?"
    ]
    for q in test_queries:
        print(f"\n🔍 Query: {q}")
        response = conditional_chain.invoke(q)
        print(f"📝 Response:\n{response}\n{'-'*80}")


🔍 Query: I am a woman and the government officer refused to issue me a citizenship certificate.
📝 Response:
**No, this is illegal.**

**Law says:**
- The National Penal (Code) Act, 2017, Section 160 (1): Except as otherwise provided by laws, no authority who exercises power under law shall, in the exercise of such power or a general law, intentionally make a discriminatory treatment against any citizen on grounds of origin, religion, color, caste, race, sex, physical condition, disability, condition of health, marital status, pregnancy, economic condition, language or region, ideology or on similar other grounds.

**Punishment:**
- A person who commits, or causes to be committed, the offence referred to in sub-section (1) shall be liable to a sentence of imprisonment for a term not exceeding three years or a fine not exceeding thirty thousand rupees or both the sentences.

**What to do:**
1. Ask for a written reason for the denial from the officer.
2. File a grievance with the head of